# Actividad: Entrenamiento con EfficientNetB0
Clasificación de calidad de limones usando el mismo dataset cargado desde Google Drive.

En esta actividad tendrá que completar los pasos faltantes en flujo del desarrollo del caso y responder las preguntas que se le realicen.

Esta actividad debera subirla en la plataforma.

Son 5 ejercicios.

***Puntuación:***

Cada ejercicio bien resuelto y sustentado: 3 puntos

Exposición: 5 puntos

In [ ]:
import os
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import image_dataset_from_directory

## Carga del Dataset desde Google Drive (.zip)

In [ ]:
!pip install gdown
import gdown
import zipfile

url = "https://drive.google.com/file/d/1-kJdc5I-vASkqjBnZpaWrZgpOSckptSx/view?usp=sharing"
file_id = url.split("/d/")[1].split("/")[0]
output = "lemon_dataset.zip"

gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

with zipfile.ZipFile(output, 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Dataset cargado en carpeta 'dataset/'")

## Cargar dataset con ImageDataset

In [ ]:
dataset_path = '/content/dataset'  # Ruta al dataset (cambiar si es otro caso)

train_ds = image_dataset_from_directory(
    os.path.join(dataset_path, 'lemon_dataset'),
    image_size=(128, 128), validation_split=0.2, subset='training',  batch_size=32, seed=42, label_mode='categorical')
val_ds = image_dataset_from_directory(
    os.path.join(dataset_path, 'lemon_dataset'),
    image_size=(128, 128), validation_split=0.2, subset='validation', batch_size=32, seed=42, label_mode='categorical')

class_names = train_ds.class_names

## Modelo EfficientNetB0 (Transfer Learning)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

### Ejercicio 1

Ayudandose de herramientas de AI Generativa explique se es lo que significa el cuadro anterior.

Respuesta:

### Ejercicio 2
En una primera iteración ejecute el modelo usando epochs = 5

En una segunda iteración ejecute el modelo usando epochs = 10

¿Qué diferencias percibe?

Respuesta:


## **Entrenamiento** del modelo

In [ ]:
# para medir el tiempo de ejecución total usaremos %%time
%%time

epochs = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

## Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(acc, label='train acc')
plt.plot(val_acc, label='val acc')
plt.legend()
plt.title('Accuracy')
plt.ylim(0.9,1)

plt.subplot(1,2,2)
plt.plot(loss, label='train loss')
plt.plot(val_loss, label='val loss')
plt.legend()
plt.title('Loss')
plt.show()

## Métricas y Matriz de Confusión

### Ejercicio 3
Calcula la matriz de confusión y las metricas en la data de validación (val_ds)

Pista: Buscar confusion_matrix o classification_report

Nota: En caso le salga error, solucionelo con ayuda de herramientas de AI Generativa como Gemini en colab o Chat GPT


In [ ]:
# COMPLETAR




### Ejercicio 4
4a. Calcula la matriz de confusión y las metricas en la data de train (train_ds)

4b. ¿El modelo esta sobreajustado?

**Sobreajuste:** Ocurre cuando un modelo aprende demasiado los detalles del conjunto de entrenamiento (como si los memorizara) y pierde capacidad para generalizar a datos nuevos.
***Ejemplo:*** En entrenamiento tiene 98% de acierto, pero en validación solo 75%.


4c. ¿Cómo interpretas el precision y el recall de la categoría de bad_quality?

In [ ]:
# COMPLETAR 4a



Respuesta:

4b.

4c.

### Ejercicio 5

¿Qué modelos en los que se uso transferlearning elegirias? Justifique.

Opción 1: MobileNetV2

Opción 2: EfficientNetB0

Respuesta:


## Inferencia sobre imágenes nuevas

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np

def predict_image(img_path):
    # 1. Cargar imagen en RGB y redimensionar igual que en el dataset
    img = load_img(img_path, target_size=(128, 128), color_mode="rgb")

    # 2. Convertir a array (0–255, float32)
    arr = img_to_array(img)           # shape: (128,128,3)

    # 3. Añadir dimensión batch, SIN dividir entre 255
    arr = np.expand_dims(arr, axis=0) # shape: (1,128,128,3)

    # 4. Predecir
    probs = model.predict(arr, verbose=0)[0]  # vector de probabilidades
    idx = np.argmax(probs)

    return class_names[idx], probs


In [ ]:
# Ejemplo:
predict_image('dataset/lemon_dataset/bad_quality/bad_quality_105.jpg')